# mine-tuning-model (RTX 5060 Ti 최적화 버전)

## 사전 준비 (터미널에서 먼저 실행)

```bash
# 1. 가상환경 생성 및 활성화
python -m venv venv
source venv/Scripts/activate

# 2. PyTorch 설치 — RTX 5060 Ti (Blackwell)는 CUDA 12.8 필요!
#    nvidia-smi 실행해서 CUDA 버전 확인 후 아래 명령어 선택

# ✅ CUDA 12.8 (RTX 5060 Ti 권장)
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

# 3. Jupyter 설치
pip install jupyter

# vs code 커널 등록
python -m ipykernel install --user --name venv --display-name "mine-tuning (venv)"

# 4. 노트북 실행
jupyter notebook
```

## 0. GPU 확인 (제일 먼저 실행!)

In [1]:
import subprocess
import sys

# GPU 확인
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
except FileNotFoundError:
    print('⚠️  nvidia-smi를 찾을 수 없습니다. NVIDIA 드라이버가 설치되어 있는지 확인하세요.')

# PyTorch CUDA 확인
import torch
print(f'PyTorch 버전: {torch.__version__}')
print(f'CUDA 버전:    {torch.version.cuda}')
print(f'CUDA 사용 가능: {torch.cuda.is_available()}')

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'GPU 이름: {gpu_name}')
    print(f'GPU VRAM: {vram_gb:.1f}GB')

    # RTX 5060 Ti는 bf16 지원 확인
    bf16_support = torch.cuda.is_bf16_supported()
    print(f'bf16 지원: {bf16_support}')  # True여야 정상
else:
    print('❌ CUDA를 사용할 수 없습니다.')
    print('   → 사전 준비 단계에서 CUDA 12.8 버전 PyTorch를 설치했는지 확인하세요.')
    sys.exit('GPU가 필요합니다.')

Wed May 20 10:24:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 591.86                 Driver Version: 591.86         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 5060 Ti   WDDM  |   00000000:01:00.0 Off |                  N/A |
| 33%   45C    P1             33W /  180W |    2830MiB /  16311MiB |    100%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. 필수 라이브러리 설치

> 💡 처음 한 번만 실행하면 돼요!

In [2]:
import subprocess
subprocess.run([
    'pip', 'install',
    'transformers==4.51.3',
    'trl==0.17.0',
    'datasets',
    'peft',
    'accelerate',
    'bitsandbytes',  # 최신 버전은 Windows 공식 지원
    '-q'
], check=True)

print('✅ 패키지 설치 완료')

✅ 패키지 설치 완료


In [3]:
!pip freeze > requirements-ft.txt

## 2. 데이터 로드 및 Instruction 형식 변환

In [4]:
from datasets import load_dataset

# 시스템 프롬프트 — 여기만 수정하면 전체 적용
SYSTEM_PROMPT = (
    "You are an expert Minecraft assistant with deep knowledge of all game mechanics, "
    "crafting recipes, biomes, mobs, and strategies.\n\n"
    "When answering:\n"
    "- Be specific and accurate about game mechanics\n"
    "- Include exact crafting recipes when relevant (materials and placement)\n"
    "- For survival tips, prioritize safety and efficiency\n"
    "- Keep answers clear and beginner-friendly unless the question is advanced"
)

def format_instruction(example):
    return {
        'text': (
            f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
            f"<|im_start|>user\n{example['question']}<|im_end|>\n"
            f"<|im_start|>assistant\n{example['answer']}<|im_end|>"
        )
    }

ds = load_dataset('ddorin/minecraft-question-answer-260k')
train_data = ds['train'].map(format_instruction, remove_columns=['question', 'answer', 'source'])

print(f'✅ 데이터 변환 완료: {len(train_data):,}개')
print('\n샘플 확인:')
print(train_data[0]['text'])


Map:   0%|          | 0/268239 [00:00<?, ? examples/s]

✅ 데이터 변환 완료: 268,239개

샘플 확인:
<|im_start|>system
You are an expert Minecraft assistant with deep knowledge of all game mechanics, crafting recipes, biomes, mobs, and strategies.

When answering:
- Be specific and accurate about game mechanics
- Include exact crafting recipes when relevant (materials and placement)
- For survival tips, prioritize safety and efficiency
- Keep answers clear and beginner-friendly unless the question is advanced<|im_end|>
<|im_start|>user
What types of leaves can be found naturally in jungle bushes in Minecraft?<|im_end|>
<|im_start|>assistant
In Minecraft, jungle bushes can generate oak leaves in the Java Edition and jungle leaves in the Bedrock Edition.<|im_end|>


## 3. 모델, 토크나이저 로드

> 💡 4bit 양자화로 로드해서 VRAM을 절약해요. 7B 모델이 약 5~6GB로 줄어들어요.

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import gc

# VRAM 비우기
gc.collect()
torch.cuda.empty_cache()

model_id = 'Qwen/Qwen2.5-7B-Instruct'

# 4bit 양자화 설정
# RTX 5060 Ti (Blackwell)는 bf16 지원 → compute_dtype을 bf16으로 설정!
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,   # ✅ 5060 Ti는 bf16 사용 (fp16보다 안정적)
    bnb_4bit_use_double_quant=True,
)

print(f'모델 로딩 중: {model_id}')
print('처음 실행 시 약 15GB 다운로드가 필요해요...')

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={'': 0},  # GPU 0번에 전체 로드
)

model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={'use_reentrant': False}
)
model.enable_input_require_grads()

used  = torch.cuda.memory_allocated() / 1024**3
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'\n✅ 모델 로드 완료!')
print(f'VRAM 사용: {used:.1f}GB / {total:.1f}GB')

모델 로딩 중: Qwen/Qwen2.5-7B-Instruct
처음 실행 시 약 15GB 다운로드가 필요해요...


W0520 10:24:43.579000 13060 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)



✅ 모델 로드 완료!
VRAM 사용: 5.2GB / 15.9GB


## 4. LoRA 어댑터 설정

> 💡 전체 모델을 학습하는 게 아니라 LoRA라는 작은 어댑터만 학습해요.
> 덕분에 VRAM을 훨씬 적게 쓰면서도 파인튜닝 효과를 얻을 수 있어요!

In [6]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4bit 학습 준비
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        'q_proj', 'k_proj',
        'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 전체 파라미터 중 약 1~2%만 학습된다고 나오면 정상이에요!

trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## 5. 학습 실행 (SFTTrainer)

### ⚙️ 매 세션마다 아래 변수만 수정하세요

In [7]:
from trl import SFTTrainer, SFTConfig
import os

# ============================================================
# 저장 경로 — 실행마다 run_001, run_002 ... 자동 증가
# ============================================================
base = r"C:\mine-tuning-output\full_train"
n = 1
while os.path.exists(f"{base}_{n:03d}"):
    n += 1
OUTPUT_DIR = f"{base}_{n:03d}"
os.makedirs(OUTPUT_DIR)

print(f"📦 전체 학습 데이터: {len(train_data):,}건")
print(f"📁 저장 경로: {OUTPUT_DIR}")

# ============================================================
# 학습 설정
# ============================================================
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,                 # 전체 데이터 1회 학습 (수렴까지는 3 권장)
    per_device_train_batch_size=4,      # RTX 5060 Ti 16GB 기준
    gradient_accumulation_steps=4,      # 실질 배치 크기 = 16
    learning_rate=2e-4,
    bf16=True,                          # RTX 5060 Ti (Blackwell) 권장
    fp16=False,                         # bf16 사용 시 반드시 False
    logging_steps=50,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    warmup_ratio=0.03,                  # 전체 스텝의 3% 워밍업
    lr_scheduler_type="cosine",
    report_to="none",
    dataloader_num_workers=0,           # Windows 멀티프로세싱 이슈 방지
    optim = paged_adamw_8bit
)

# ============================================================
# Trainer 생성 및 학습
# ============================================================
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    args=training_args,
    processing_class=tokenizer,
)

print("🚀 학습 시작!")
trainer.train()

# 최종 저장
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\n✅ 학습 완료! → {OUTPUT_DIR}")


📦 전체 학습 데이터: 268,239건
📁 저장 경로: C:\mine-tuning-output\full_train_001


Converting train dataset to ChatML:   0%|          | 0/268239 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/268239 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/268239 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/268239 [00:00<?, ? examples/s]

...

🚀 학습 시작!


`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
50,2.537400
100,0.787900
150,0.677700
200,0.658100
250,0.633800
300,0.627100
350,0.626600
400,0.609500
450,0.594100
500,0.597100


c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\SSAFY\Desktop\ㄷㅇ\mine-tuning-model\venv\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two var


✅ 학습 완료! → C:\mine-tuning-output\full_train_001


## 6. 버전 확인

In [8]:
import transformers, trl, torch

print(f'transformers: {transformers.__version__}')
print(f'trl:          {trl.__version__}')
print(f'torch:        {torch.__version__}')
print(f'CUDA:         {torch.version.cuda}')

transformers: 4.51.3
trl:          0.17.0
torch:        2.11.0+cu128
CUDA:         12.8
